In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l3.4 Multi-head attention

> 🎯 **Goal:** Run several attention computations in parallel on slices of the channels, then recombine them into one tensor.

**Why this matters.** A single attention head can only learn one kind of relationship at a time. Real language has many overlapping structures at once: grammar, who-did-what-to-whom, nearby word order, long-range topic. Multi-head attention is how a transformer tracks several of these simultaneously without making each one fight for the same parameters.

**The intuition.** Picture several people each reading the same sentence, but each looking for a different thing: one tracks the grammar, one tracks the subject, one tracks the tone, one tracks position. They read in parallel and then pool their notes. Each "person" is a head, and each gets its own slice of the channels to work with.

**The mechanics.** We have `channels = n_head * hs`, so the channel dimension splits evenly into `n_head` heads of size `hs` each. The reshape does the splitting:
- `to_heads` reshapes `[1, tokens, channels]` into `[1, tokens, n_head, hs]`, then transposes to `[1, n_head, tokens, hs]`. Now the head dimension sits in front of tokens, so attention can run on all heads at once as a batched operation.
- `scaled_dot_product_attention` runs independently per head; each head attends within its own `hs`-sized subspace, never mixing slices.
- After attention we reverse the reshape: transpose back, `contiguous()`, and `view` to glue the heads' outputs side by side into `[1, tokens, channels]`.

Crucially each head still only sees its own slice of the channels, and the heads run truly in parallel as extra batch dimensions, not in a Python loop.

In [ ]:
from pocketlm import scaled_dot_product_attention

torch.manual_seed(0)
tokens, n_head, hs = 5, 4, 6
channels = n_head * hs

def to_heads(t):
    return t.view(1, tokens, n_head, hs).transpose(1, 2)   # [1, n_head, T, hs]

q = to_heads(torch.randn(1, tokens, channels))
k = to_heads(torch.randn(1, tokens, channels))
v = to_heads(torch.randn(1, tokens, channels))
out, w = scaled_dot_product_attention(q, k, v, causal=True)
combined = out.transpose(1, 2).contiguous().view(1, tokens, channels)
print("per-head out:", tuple(out.shape), " recombined:", tuple(combined.shape))

**What just happened.** The printout shows the per-head output as `[1, n_head, tokens, hs]` (four heads each producing a `tokens x hs` result) and the recombined tensor as `[1, tokens, channels]`. That recombined shape is identical to what we started with, so the whole multi-head machinery is invisible from outside the block: it takes in `[batch, tokens, channels]` and hands back `[batch, tokens, channels]`. The split into heads, the parallel attention, and the merge were an internal detour. The next layer neither knows nor cares that four heads were involved.

⚠️ **Common pitfalls**
- Forgetting `contiguous()` before `view`. After a `transpose`, the tensor's memory layout is non-contiguous, and `view` will raise an error. `contiguous()` materializes a clean copy first.
- Mismatched split. `channels` must equal `n_head * hs` exactly. If it does not, the `view` either errors or silently scrambles channels across heads.
- Recombining in the wrong order. You must transpose back *before* the final `view`, otherwise the heads interleave incorrectly and the channels come out shuffled.

🏋️ **Try it yourself.** Change `n_head` from 4 to 2 (keeping `hs` at 6, so `channels` becomes 12, then 24) and rerun. The per-head shape changes but the recombined shape always lands back at `[1, tokens, channels]`. Try `n_head=6, hs=1` versus `n_head=1, hs=6`: same channel budget, very different attention behavior.

In [ ]:
assert tuple(out.shape) == (1, n_head, tokens, hs)
assert tuple(w.shape) == (1, n_head, tokens, tokens)
assert tuple(combined.shape) == (1, tokens, channels)